# Exploratory Analysis and Model Validation

This notebook validates the main analytical components of **Swiss Financial Health Advisor**.

It is intentionally written as a portfolio-facing notebook: the goal is not only to show charts, but to explain whether the synthetic data, scoring logic, segmentation, and recommendation engine behave coherently.

## Validation Questions

1. Does the synthetic dataset contain enough behavioral variety?
2. Are the engineered financial features plausible?
3. Does the Financial Health Score produce interpretable bands?
4. Does K-Means recover meaningful user segments?
5. Are recommendations aligned with each user's weakest score dimension?

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from swiss_financial_health.clustering import CLUSTER_FEATURES

DATA = ROOT / "data"

In [ ]:
transactions = pd.read_csv(DATA / "synthetic_transactions.csv", parse_dates=["date"])
monthly = pd.read_csv(DATA / "user_monthly_features.csv", parse_dates=["month"])
users = pd.read_csv(DATA / "scored_users.csv")
recommendations = pd.read_csv(DATA / "recommendations.csv")
centroids = pd.read_csv(DATA / "segment_centroids.csv")

print("transactions", transactions.shape)
print("monthly", monthly.shape)
print("users", users.shape)
print("recommendations", recommendations.shape)
print("centroids", centroids.shape)

## 1. Synthetic Data Coverage

The generator creates multiple behavioral archetypes: stable savers, balanced users, high discretionary spenders, variable-income users, and debt-pressure users.

The profile labels are used only to validate the synthetic design. The segmentation model itself does not receive the label as an input feature.

In [ ]:
profile_counts = users["profile"].value_counts().rename_axis("profile").reset_index(name="users")
profile_counts

In [ ]:
fig = px.bar(
    profile_counts,
    x="profile",
    y="users",
    title="Synthetic Users by Generator Profile",
    text="users",
)
fig.update_layout(xaxis_title="Generator profile", yaxis_title="Users")
fig.show()

In [ ]:
monthly_summary = monthly[["income", "expense", "savings", "debt_payments", "discretionary_spend"]].describe().round(2)
monthly_summary

## 2. Financial Health Score Distribution

A useful portfolio prototype should avoid scores that collapse into one narrow range. The distribution below checks whether the composite score produces differentiated outcomes across users.

In [ ]:
users[[
    "financial_health_score",
    "liquidity_score",
    "income_stability_score",
    "savings_score",
    "debt_score",
]].describe().round(2)

In [ ]:
fig = px.histogram(
    users,
    x="financial_health_score",
    color="health_band",
    nbins=25,
    title="Financial Health Score Distribution",
)
fig.update_layout(xaxis_title="Financial Health Score", yaxis_title="Users")
fig.show()

In [ ]:
fig = px.box(
    users,
    x="profile",
    y="financial_health_score",
    color="profile",
    title="Score Distribution by Synthetic Profile",
)
fig.update_layout(xaxis_title="Generator profile", yaxis_title="Financial Health Score", showlegend=False)
fig.show()

## 3. Feature Relationships

The score should respond to interpretable financial behavior. In this prototype, higher savings rates should generally improve the score, while higher debt exposure should generally reduce it.

In [ ]:
correlations = users[[
    "financial_health_score",
    "avg_savings_rate",
    "avg_debt_to_income",
    "avg_discretionary_ratio",
    "income_cv",
    "avg_net_cashflow",
]].corr()["financial_health_score"].sort_values(ascending=False).round(3)
correlations

In [ ]:
fig = px.scatter(
    users,
    x="avg_savings_rate",
    y="avg_debt_to_income",
    color="financial_health_score",
    size="avg_income",
    hover_data=["user_id", "profile", "health_band"],
    title="Savings Capacity vs. Debt Exposure",
    color_continuous_scale="RdYlGn",
)
fig.update_layout(xaxis_title="Average savings rate", yaxis_title="Debt-to-income")
fig.show()

## 4. Segmentation Validation

K-Means is evaluated with silhouette score. In a real implementation, this section would compare multiple values of `k` and validate stability across time windows.

For this MVP, we validate that the current five-segment configuration produces separable and business-readable clusters.

In [ ]:
X = users[CLUSTER_FEATURES]
X_scaled = StandardScaler().fit_transform(X)
labels = users["segment"]
current_silhouette = silhouette_score(X_scaled, labels)
print(f"Current silhouette score: {current_silhouette:.3f}")

In [ ]:
centroids.sort_values("financial_health_score", ascending=False)

In [ ]:
segment_profile_matrix = pd.crosstab(users["segment_label"], users["profile"], normalize="index").round(2)
segment_profile_matrix

In [ ]:
fig = px.scatter(
    users,
    x="avg_savings_rate",
    y="avg_debt_to_income",
    color="segment_label",
    size="financial_health_score",
    hover_data=["user_id", "profile", "health_band"],
    title="Behavioral Segments in Feature Space",
)
fig.update_layout(xaxis_title="Average savings rate", yaxis_title="Debt-to-income")
fig.show()

## 5. Recommendation Alignment

The recommendation engine is rule-based: it identifies the weakest score dimension and maps it to a human-readable action.

The validation below checks whether every user receives exactly one recommendation and whether each priority dimension is distributed plausibly across the population.

In [ ]:
recommendation_check = users[["user_id", "liquidity_score", "income_stability_score", "savings_score", "debt_score"]].merge(
    recommendations,
    on="user_id",
    how="left",
)

print("Users without recommendation:", recommendation_check["recommendation"].isna().sum())
recommendations["priority_dimension"].value_counts().rename_axis("priority_dimension").reset_index(name="users")

In [ ]:
fig = px.bar(
    recommendations["priority_dimension"].value_counts().rename_axis("priority_dimension").reset_index(name="users"),
    x="priority_dimension",
    y="users",
    title="Recommendation Priority Distribution",
    text="users",
)
fig.update_layout(xaxis_title="Priority dimension", yaxis_title="Users")
fig.show()

## Validation Summary

The current MVP is coherent for a portfolio project:

- The synthetic data generator produces distinct financial behavior patterns.
- The composite score creates differentiated and explainable user outcomes.
- The clustering layer produces business-readable segments with a useful silhouette score for synthetic data.
- The recommendation engine is transparent and aligned with the weakest score dimension.

## Limitations

- The data is synthetic and cannot validate real-world behavioral assumptions.
- K-Means is sensitive to feature scaling and the chosen number of clusters.
- The score weights are expert-defined, not empirically optimized.
- Recommendations are deterministic rules, not personalized causal interventions.

## Next Analytical Improvements

- Compare multiple cluster counts and include an elbow/silhouette curve.
- Add sensitivity analysis for score weights.
- Simulate multi-bank Open Banking aggregation.
- Track score evolution over time and detect early deterioration.
- Add fairness checks if demographic or protected-class proxies are ever introduced.